In [10]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, accuracy_score, cohen_kappa_score
from numpy.lib.stride_tricks import sliding_window_view
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

In [11]:
def load_raster(path):
    """Load a single-band raster and return data, transform, meta."""
    with rasterio.open(path) as src:
        return src.read(1).astype(np.float32), src.transform, src.meta

def get_fast_neighborhood(lulc_map, window_size=3):
    """Extract neighborhood features using sliding windows (CA component)."""
    padding = window_size // 2
    padded = np.pad(lulc_map, padding, mode='edge')
    windows = sliding_window_view(padded, (window_size, window_size))
    return windows.reshape(-1, window_size * window_size)


In [12]:
BASE = "."

lulc_2005_raw, transform_2005, meta_2005 = load_raster(f"{BASE}/classified2005.tif")
lulc_2015_raw, transform_2015, meta_2015 = load_raster(f"{BASE}/classified_2015.tif")
lulc_2025_raw, transform_2025, meta_2025 = load_raster(f"{BASE}/classified2025.tif")

In [13]:
# FIX Issue #1 & #2: Align dimensions and handle nodata
# 2005 is (2069,2269) while others are (2068,2268). Crop to common size.
target_h, target_w = 2068, 2268
lulc_2005 = lulc_2005_raw[:target_h, :target_w].copy()
lulc_2015 = lulc_2015_raw[:target_h, :target_w].copy()
lulc_2025 = lulc_2025_raw[:target_h, :target_w].copy()
# FIX Issue #2: Create valid mask (255 = nodata in 2005)
valid_mask = (lulc_2005 != 255) & (lulc_2005 >= 1) & (lulc_2005 <= 4)
valid_mask &= (lulc_2015 >= 1) & (lulc_2015 <= 4)
valid_mask &= (lulc_2025 >= 1) & (lulc_2025 <= 4)
# Set nodata pixels to 0 (will be excluded from training)
lulc_2005[~valid_mask] = 0
lulc_2015[~valid_mask] = 0
lulc_2025[~valid_mask] = 0
print(f"Raster shape: {target_h} x {target_w}")
print(f"Valid pixels: {valid_mask.sum():,} / {target_h * target_w:,} ({valid_mask.mean()*100:.1f}%)")
print(f"LULC classes: {np.unique(lulc_2015[valid_mask])}")
# Classes: 1=Developed, 2=Vegetation, 3=Wetland, 4=Water


Raster shape: 2068 x 2268
Valid pixels: 4,690,224 / 4,690,224 (100.0%)
LULC classes: [1. 2. 3. 4.]


In [14]:
# %% Cell 4: Load and normalize driver variables
driver_paths = [
    f'{BASE}/Elevation.tif',
    f'{BASE}/Slope.tif',
    f'{BASE}/Distance_metro.tif',
    f'{BASE}/EucDist_PrimaryRoad.tif',
    f'{BASE}/EucDist_railwaySt.tif',
    f'{BASE}/EucDist_roadnetwork.tif',
    f'{BASE}/EucDist_secondaryRoads.tif',
    f'{BASE}/EucDist_waterbody.tif',
    f'{BASE}/population.tif',
    f'{BASE}/NDVI.tif',
    f'{BASE}/Precipitation.tif'
]
driver_names = ['Elevation', 'Slope', 'Dist_Metro', 'Dist_PrimaryRd',
                'Dist_Railway', 'Dist_RoadNet', 'Dist_SecondaryRd',
                'Dist_Waterbody', 'Population', 'NDVI', 'Precipitation']
driver_data = []
for p in driver_paths:
    d, _, _ = load_raster(p)
    d = d[:target_h, :target_w]
    # Replace nodata with 0 for distance layers (negative nodata)
    d[d < 0] = 0
    driver_data.append(d)
# Stack and normalize to [0, 1] as per paper methodology
driver_stack = np.stack(driver_data, axis=-1)  # (H, W, 11)
h, w, n_drivers = driver_stack.shape
scaler = MinMaxScaler(feature_range=(0, 1))
driver_flat = driver_stack.reshape(-1, n_drivers)
driver_norm = scaler.fit_transform(driver_flat).reshape(h, w, n_drivers)
print(f"Driver stack shape: {driver_stack.shape}")
print(f"Drivers: {driver_names}")


Driver stack shape: (2068, 2268, 11)
Drivers: ['Elevation', 'Slope', 'Dist_Metro', 'Dist_PrimaryRd', 'Dist_Railway', 'Dist_RoadNet', 'Dist_SecondaryRd', 'Dist_Waterbody', 'Population', 'NDVI', 'Precipitation']


In [15]:
# %% Cell 5: Compute Transition Probability Matrix (2005 → 2015)
# FIX Issue #6: This is critical for the CA model
def compute_transition_matrix(lulc_from, lulc_to, mask, n_classes=4):
    """Compute Markov chain transition probability matrix."""
    from_vals = lulc_from[mask].astype(int)
    to_vals = lulc_to[mask].astype(int)

    tpm = np.zeros((n_classes, n_classes))
    for i in range(1, n_classes + 1):
        for j in range(1, n_classes + 1):
            tpm[i-1, j-1] = np.sum((from_vals == i) & (to_vals == j))

    # Normalize rows to get probabilities
    row_sums = tpm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1  # avoid division by zero
    tpm = tpm / row_sums
    return tpm
tpm = compute_transition_matrix(lulc_2005, lulc_2015, valid_mask)
class_names = ['Developed', 'Vegetation', 'Wetland', 'Water']
print("\nTransition Probability Matrix (2005 → 2015):")
tpm_df = pd.DataFrame(tpm, index=class_names, columns=class_names)
print(tpm_df.round(4))



Transition Probability Matrix (2005 → 2015):
            Developed  Vegetation  Wetland   Water
Developed      0.6501      0.1196   0.0098  0.2205
Vegetation     0.0536      0.6854   0.0091  0.2519
Wetland        0.0191      0.1919   0.7558  0.0333
Water          0.0976      0.2442   0.0064  0.6518


In [16]:
#  %% Cell 6: LULC Change Analysis (Dynamic Degree)
def compute_lulc_changes(lulc_from, lulc_to, mask, year_from, year_to, class_names):
    """Compute area changes and Dynamic Degree (annual rate of change)."""
    t = year_to - year_from
    results = []
    for cls_id in range(1, len(class_names) + 1):
        area_from = np.sum((lulc_from[mask] == cls_id))
        area_to = np.sum((lulc_to[mask] == cls_id))
        delta = area_to - area_from
        dd = (delta / area_from * 100 / t) if area_from > 0 else 0
        results.append({
            'Class': class_names[cls_id - 1],
            f'Pixels_{year_from}': area_from,
            f'Pixels_{year_to}': area_to,
            'Delta_Pixels': delta,
            'DD_%_per_yr': round(dd, 3)
        })
    return pd.DataFrame(results)
print("\n=== LULC Changes 2005-2015 ===")
changes_1 = compute_lulc_changes(lulc_2005, lulc_2015, valid_mask, 2005, 2015, class_names)
print(changes_1.to_string(index=False))
print("\n=== LULC Changes 2015-2025 ===")
changes_2 = compute_lulc_changes(lulc_2015, lulc_2025, valid_mask, 2015, 2025, class_names)
print(changes_2.to_string(index=False))


=== LULC Changes 2005-2015 ===
     Class  Pixels_2005  Pixels_2015  Delta_Pixels  DD_%_per_yr
 Developed       501258       646342        145084        2.894
Vegetation      1962946      1947682        -15264       -0.078
   Wetland        24967        55788         30821       12.345
     Water      2201053      2040412       -160641       -0.730

=== LULC Changes 2015-2025 ===
     Class  Pixels_2015  Pixels_2025  Delta_Pixels  DD_%_per_yr
 Developed       646342      1120901        474559        7.342
Vegetation      1947682      1824877       -122805       -0.631
   Wetland        55788        71005         15217        2.728
     Water      2040412      1673441       -366971       -1.799


In [17]:
# %% Cell 7: Prepare training data with proper masking
WINDOW_SIZE = 5 # 5x5 Moore neighborhood for CA
# Get neighborhood features from 2005 map (input to predict 2015)
X_nb_all = get_fast_neighborhood(lulc_2005, window_size=WINDOW_SIZE)
X_drivers_all = driver_norm.reshape(-1, n_drivers)
# Combine drivers + neighborhood = feature vector per pixel
X_all = np.hstack([X_drivers_all, X_nb_all])
# Target: 2015 LULC
y_all = lulc_2015.flatten()
# Only use valid pixels for training
valid_flat = valid_mask.flatten()
X_valid = X_all[valid_flat]
y_valid = y_all[valid_flat]
print(f"Total features per pixel: {X_all.shape[1]} ({n_drivers} drivers + {WINDOW_SIZE**2} neighborhood)")
print(f"Valid training pixels: {len(y_valid):,}")
print(f"Class distribution: {dict(zip(*np.unique(y_valid, return_counts=True)))}")

Total features per pixel: 36 (11 drivers + 25 neighborhood)
Valid training pixels: 4,690,224
Class distribution: {np.float32(1.0): np.int64(646342), np.float32(2.0): np.int64(1947682), np.float32(3.0): np.int64(55788), np.float32(4.0): np.int64(2040412)}


In [18]:
# %% Cell 8: Stratified sampling for balanced training
def get_stratified_samples(X, y, samples_per_class=25000, seed=42):
    """Sample equal number of points per class for balanced training."""
    np.random.seed(seed)
    unique_classes = np.unique(y)
    indices = []
    for cls in unique_classes:
        if cls == 0:  # skip nodata
            continue
        cls_idx = np.where(y == cls)[0]
        n = min(len(cls_idx), samples_per_class)
        selected = np.random.choice(cls_idx, n, replace=False)
        indices.extend(selected)
    np.random.shuffle(indices)
    return X[indices], y[indices]
X_train, y_train = get_stratified_samples(X_valid, y_valid, samples_per_class=25000)
print(f"Balanced training set: {len(y_train):,} samples")
print(f"Per class: {dict(zip(*np.unique(y_train, return_counts=True)))}")

Balanced training set: 100,000 samples
Per class: {np.float32(1.0): np.int64(25000), np.float32(2.0): np.int64(25000), np.float32(3.0): np.int64(25000), np.float32(4.0): np.int64(25000)}


In [19]:
# %% Cell 9: Train ANN (MLP Classifier)
ann_model = MLPClassifier(
    hidden_layer_sizes=(128, 128, 64),  # 3 hidden layers
    activation='relu',
    learning_rate_init=0.001,
    momentum=0.9,
    max_iter=1000,
    early_stopping=True,         # Use validation-based early stopping
    validation_fraction=0.15,    # 15% held out for validation
    n_iter_no_change=15,         # Patience
    random_state=42,
    verbose=True
)
print("Training ANN...")
ann_model.fit(X_train, y_train)
print(f"\nTraining complete! Best validation score: {ann_model.best_validation_score_:.4f}")
print(f"Iterations: {ann_model.n_iter_}")


Training ANN...
Iteration 1, loss = 1.12992129
Validation score: 0.576600
Iteration 2, loss = 0.99430509
Validation score: 0.605000
Iteration 3, loss = 0.95398054
Validation score: 0.603333
Iteration 4, loss = 0.93266807
Validation score: 0.609467
Iteration 5, loss = 0.91062067
Validation score: 0.630667
Iteration 6, loss = 0.89634164
Validation score: 0.635600
Iteration 7, loss = 0.88043890
Validation score: 0.648267
Iteration 8, loss = 0.86889695
Validation score: 0.648933
Iteration 9, loss = 0.85599246
Validation score: 0.648067
Iteration 10, loss = 0.84704155
Validation score: 0.653600
Iteration 11, loss = 0.83786216
Validation score: 0.658400
Iteration 12, loss = 0.82669420
Validation score: 0.657600
Iteration 13, loss = 0.82028553
Validation score: 0.648067
Iteration 14, loss = 0.81611225
Validation score: 0.658800
Iteration 15, loss = 0.80568143
Validation score: 0.670133
Iteration 16, loss = 0.79750167
Validation score: 0.664533
Iteration 17, loss = 0.79082866
Validation score:

In [20]:
# %% Cell 9: Train ANN (MLP Classifier)
ann_model = MLPClassifier(
    hidden_layer_sizes=(128, 128, 64),  # 3 hidden layers
    activation='relu',
    learning_rate_init=0.001,
    momentum=0.9,
    max_iter=1000,
    early_stopping=True,         # Use validation-based early stopping
    validation_fraction=0.15,    # 15% held out for validation
    n_iter_no_change=15,         # Patience
    random_state=42,
    verbose=True
)
print("Training ANN...")
ann_model.fit(X_train, y_train)
print(f"\nTraining complete! Best validation score: {ann_model.best_validation_score_:.4f}")
print(f"Iterations: {ann_model.n_iter_}")


Training ANN...
Iteration 1, loss = 1.12992129
Validation score: 0.576600
Iteration 2, loss = 0.99430509
Validation score: 0.605000
Iteration 3, loss = 0.95398054
Validation score: 0.603333
Iteration 4, loss = 0.93266807
Validation score: 0.609467
Iteration 5, loss = 0.91062067
Validation score: 0.630667
Iteration 6, loss = 0.89634164
Validation score: 0.635600
Iteration 7, loss = 0.88043890
Validation score: 0.648267
Iteration 8, loss = 0.86889695
Validation score: 0.648933
Iteration 9, loss = 0.85599246
Validation score: 0.648067
Iteration 10, loss = 0.84704155
Validation score: 0.653600
Iteration 11, loss = 0.83786216
Validation score: 0.658400
Iteration 12, loss = 0.82669420
Validation score: 0.657600
Iteration 13, loss = 0.82028553
Validation score: 0.648067
Iteration 14, loss = 0.81611225
Validation score: 0.658800
Iteration 15, loss = 0.80568143
Validation score: 0.670133
Iteration 16, loss = 0.79750167
Validation score: 0.664533
Iteration 17, loss = 0.79082866
Validation score:

In [21]:
# %% Cell 10: CA-ANN Simulation function
# FIX Issue #4: Proper iterative CA simulation with transition constraints
def ca_ann_simulate(ann_model, lulc_current, driver_norm, tpm, valid_mask,
                    n_iterations=10, window_size=3, stochastic_perturbation=0.1):
    """
    Iterative CA-ANN simulation.

    The CA component provides spatial context (neighborhood),
    the ANN provides transition potential from drivers,
    and the TPM constrains which transitions are allowed.
    """
    h, w = lulc_current.shape
    simulated = lulc_current.copy()
    n_drivers = driver_norm.shape[-1]

    for iteration in range(n_iterations):
        # 1. Extract neighborhood from current state (CA part)
        nb_features = get_fast_neighborhood(simulated, window_size=window_size)

        # 2. Combine with drivers
        X_sim = np.hstack([driver_norm.reshape(-1, n_drivers), nb_features])

        # 3. Get transition probabilities from ANN
        ann_probs = ann_model.predict_proba(X_sim)  # (n_pixels, n_classes)
        ann_classes = ann_model.classes_

        # 4. Apply transition constraints from TPM
        current_flat = simulated.flatten()
        new_flat = current_flat.copy()
        valid_flat = valid_mask.flatten()

        for idx in range(len(current_flat)):
            if not valid_flat[idx]:
                continue

            current_class = int(current_flat[idx])
            if current_class < 1 or current_class > 4:
                continue

            # Get allowed transition probabilities from Markov chain
            tp = tpm[current_class - 1]  # transition probs for this class

            # Get ANN-predicted probabilities
            ann_p = np.zeros(4)
            for k, c in enumerate(ann_classes):
                if 1 <= c <= 4:
                    ann_p[int(c) - 1] = ann_probs[idx, k]

            # Combined probability = TPM * ANN (joint probability)
            combined = tp * ann_p

            # Add stochastic perturbation (simulates uncertainty)
            noise = 1 + stochastic_perturbation * (np.random.random(4) - 0.5)
            combined *= noise

            # Normalize
            if combined.sum() > 0:
                combined /= combined.sum()
                new_flat[idx] = np.argmax(combined) + 1  # classes are 1-indexed

        simulated = new_flat.reshape(h, w)

        # Count changes this iteration
        changes = np.sum((current_flat != new_flat) & valid_flat)
        print(f"  Iteration {iteration+1}/{n_iterations}: {changes:,} pixels changed")

        if changes == 0:
            print("  Converged - no more changes.")
            break

    return simulated

In [22]:
# %% Cell 11: Validate - Simulate 2025 from 2015 context
print("=== VALIDATION: Simulating 2025 from 2015 ===")
# Get neighborhood from 2015 (the "current state" for prediction)
pred_2025 = ca_ann_simulate(
    ann_model=ann_model,
    lulc_current=lulc_2015,
    driver_norm=driver_norm,
    tpm=tpm,
    valid_mask=valid_mask,
    n_iterations=5,       # fewer iterations for validation
    window_size=WINDOW_SIZE,
    stochastic_perturbation=0.05
)
print("Simulation complete!")


=== VALIDATION: Simulating 2025 from 2015 ===
  Iteration 1/5: 195,163 pixels changed
  Iteration 2/5: 115,154 pixels changed
  Iteration 3/5: 75,750 pixels changed
  Iteration 4/5: 54,527 pixels changed
  Iteration 5/5: 42,224 pixels changed
Simulation complete!


In [23]:
# %% Cell 12: Accuracy Assessment (Pontius & Millones method)
def calculate_detailed_accuracy(y_true, y_pred, class_names, mask_flat):
    """Full accuracy assessment following the paper's methodology."""
    # Filter to valid pixels only
    valid_idx = mask_flat
    yt = y_true[valid_idx].astype(int)
    yp = y_pred[valid_idx].astype(int)

    # Standard metrics
    oa = accuracy_score(yt, yp)
    kappa = cohen_kappa_score(yt, yp)

    # Confusion matrix
    cm = confusion_matrix(yt, yp, labels=[1, 2, 3, 4])

    # Producer & User accuracy
    col_sums_cm = cm.sum(axis=0)
    row_sums_cm = cm.sum(axis=1)
    producer_acc = np.where(col_sums_cm > 0, np.diag(cm) / col_sums_cm * 100, 0)
    user_acc = np.where(row_sums_cm > 0, np.diag(cm) / row_sums_cm * 100, 0)

    # Pontius & Millones disagreement
    n = cm.sum()
    p = cm / n
    p_row = p.sum(axis=1)
    p_col = p.sum(axis=0)
    quantity_disagreement = np.sum(np.abs(p_row - p_col)) / 2

    p_diag = np.diag(p)
    omission = p_col - p_diag
    commission = p_row - p_diag
    allocation_disagreement = np.sum(np.minimum(omission, commission))

    print("=" * 50)
    print("ACCURACY ASSESSMENT")
    print("=" * 50)
    print(f"Overall Agreement:        {oa*100:.2f}%")
    print(f"Overall Kappa:            {kappa:.4f}")
    print(f"Quantity Disagreement:    {quantity_disagreement*100:.2f}%")
    print(f"Allocation Disagreement:  {allocation_disagreement*100:.2f}%")
    print(f"Total Disagreement:       {(quantity_disagreement+allocation_disagreement)*100:.2f}%")

    print(f"\nConfusion Matrix:")
    print(pd.DataFrame(cm, index=class_names, columns=class_names))

    reliability_df = pd.DataFrame({
        'Class': class_names,
        'Producer Acc (%)': producer_acc.round(2),
        'User Acc (%)': user_acc.round(2)
    })
    print(f"\nClass Reliability:")
    print(reliability_df.to_string(index=False))

    return oa, kappa
oa, kappa = calculate_detailed_accuracy(
    lulc_2025.flatten(), pred_2025.flatten(), class_names, valid_mask.flatten()
)

ACCURACY ASSESSMENT
Overall Agreement:        61.56%
Overall Kappa:            0.4141
Quantity Disagreement:    6.26%
Allocation Disagreement:  32.18%
Total Disagreement:       38.44%

Confusion Matrix:
            Developed  Vegetation  Wetland    Water
Developed      611361      194659     5697   309184
Vegetation     103600     1192750    23836   504691
Wetland          4920       16984    34401    14700
Water          107251      508968     8556  1048666

Class Reliability:
     Class  Producer Acc (%)  User Acc (%)
 Developed             73.91         54.54
Vegetation             62.34         65.36
   Wetland             47.46         48.45
     Water             55.86         62.67
